In [3]:
pip install datasets transformers

In [4]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [5]:
import json
import os
import pandas as pd
import re
from html import unescape
from pathlib import Path
import torch

# ✅ Enable GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

from datasets import load_dataset
from transformers import (
    AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments, pipeline
)

# ✅ Function to clean text
def clean_text(text):
    """Cleans text by handling HTML entities and normalizing whitespace."""
    text = unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ✅ Load Dataset
file_path = Path("dev.json")
jsonl_output_file = "gpt_qa_dataset.jsonl"

# ✅ Extract QA pairs
def load_and_process_data(file_path):
    """Loads dataset and extracts context, question, and answer pairs."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    qa_pairs = set()
    for article in data.get('data', []):
        for paragraph in article.get('paragraphs', []):
            context = clean_text(paragraph.get('context', ''))
            for qa in paragraph.get('qas', []):
                question = clean_text(qa.get('question', ''))
                answers = qa.get('answers', [])
                answer_text = clean_text(answers[0].get('text', '')) if answers else 'No Answer'
                qa_pairs.add((context, question, answer_text))

    df = pd.DataFrame(qa_pairs, columns=['context', 'question', 'answer'])
    df.to_json(jsonl_output_file, orient="records", lines=True)  # Save as JSONL
    return jsonl_output_file

jsonl_output_file = load_and_process_data(file_path)

# ✅ Load Dataset for Training
dataset = load_dataset("json", data_files=jsonl_output_file)
dataset = dataset["train"].train_test_split(test_size=0.1)

# ✅ Load Pretrained Model
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

# ✅ Preprocess Data with Correct Token Positions
def preprocess_function(examples):
    """Tokenizes and aligns token positions correctly."""
    inputs = tokenizer(
        examples["question"], examples["context"],
        truncation=True, padding="max_length", max_length=512,
        return_offsets_mapping=True  # Enables mapping between text and token positions
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples["answer"])):
        answer = examples["answer"][i]
        context = examples["context"][i]

        start_idx = context.find(answer)
        end_idx = start_idx + len(answer)

        offsets = inputs["offset_mapping"][i]

        token_start = token_end = None
        for idx, (start, end) in enumerate(offsets):
            if start_idx >= start and start_idx < end:
                token_start = idx
            if end_idx > start and end_idx <= end:
                token_end = idx

        if token_start is None:
            token_start = 0  # CLS token fallback
        if token_end is None:
            token_end = 0  # CLS token fallback

        start_positions.append(token_start)
        end_positions.append(token_end)

    inputs.pop("offset_mapping")  # Remove unused offset mapping
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs


tokenized_datasets = dataset.map(preprocess_function, batched=True)

# ✅ Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    fp16=True,
)

# ✅ Train Model if Not Already Trained
if not os.path.exists("./fine_tuned_qa_model"):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
    )
    trainer.train()
    trainer.save_model("./fine_tuned_qa_model")
    tokenizer.save_pretrained("./fine_tuned_qa_model")
    print("✅ Model trained and saved!")
else:
    print("✅ Using existing trained model!")

# ✅ Load Fine-Tuned Model for Testing
qa_pipeline = pipeline("question-answering", model="./fine_tuned_qa_model", tokenizer="./fine_tuned_qa_model", device=0)

# ✅ Test Model
examples = [
    {"context": "Cristiano Ronaldo has won five Ballon d'Or awards.", "question": "How many Ballon d'Or awards has Cristiano Ronaldo won?"},
    {"context": "The Great Wall of China is over 13,000 miles long.", "question": "How long is the Great Wall of China?"},
    {"context": "Barack Obama was the 44th President of the United States.", "question": "Who was the 44th President of the United States?"},
]

for example in examples:
    result = qa_pipeline(question=example["question"], context=example["context"])
    print(f"🔹 Question: {example['question']}")
    print(f"✅ Predicted Answer: {result['answer']}\n")


Using device: cuda


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Device set to use cuda:0


✅ Using existing trained model!
🔹 Question: How many Ballon d'Or awards has Cristiano Ronaldo won?
✅ Predicted Answer: five

🔹 Question: How long is the Great Wall of China?
✅ Predicted Answer: 13,000 miles

🔹 Question: Who was the 44th President of the United States?
✅ Predicted Answer: Barack Obama



In [6]:
pip install fastapi uvicorn

In [7]:
!pip install pyngrok

In [8]:
!pkill -f ngrok
!pkill -f uvicorn


In [9]:
!ngrok authtoken 2srxaiIvk07knEiGUgRIh9eLAQG_6gqmoFZH2sZk9uKi3fUwb

from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f"✅ New public URL: {public_url}")



Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ New public URL: NgrokTunnel: "https://7e94-34-143-236-222.ngrok-free.app" -> "http://localhost:8000"


In [10]:
!curl -X GET http://eccb-34-143-236-222.ngrok-free.app/docs


<a href="https://eccb-34-143-236-222.ngrok-free.app/docs">Temporary Redirect</a>.



In [11]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline
import threading

app = FastAPI()

# ✅ Load Fine-Tuned QA Model
qa_pipeline = pipeline("question-answering", model="./fine_tuned_qa_model", tokenizer="./fine_tuned_qa_model", device=0)

class QARequest(BaseModel):
    question: str
    context: str

@app.post("/answer/")
def get_answer(request: QARequest):
    result = qa_pipeline(question=request.question, context=request.context)
    return {"answer": result["answer"]}

# ✅ Apply fix for Jupyter Notebook event loop
nest_asyncio.apply()

# ✅ Function to start FastAPI
def start_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# ✅ Run API in a background thread
api_thread = threading.Thread(target=start_fastapi)
api_thread.start()


Device set to use cuda:0


In [12]:
import requests

url = "http://eccb-34-143-236-222.ngrok-free.app/answer/"
data = {
    "question": "Who is the CEO of Tesla?",
    "context": "Elon Musk is the CEO of Tesla."
}

response = requests.post(url, json=data)
print("Status Code:", response.status_code)
print("Response:", response.text)



Status Code: 404
Response: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-Semibold-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://cdn.ngrok.com/static/fonts/ib

In [13]:
pip install gradio

INFO:     Started server process [3407]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [16]:
import gradio as gr
from transformers import pipeline

# ✅ Load Fine-Tuned Model
qa_pipeline = pipeline("question-answering", model="./fine_tuned_qa_model", tokenizer="./fine_tuned_qa_model", device=0)

# ✅ Function to Get Answer
def answer_question(context, question):
    if not context or not question:
        return "Please provide both a context and a question."
    try:
        result = qa_pipeline(question=question, context=context)
        return result['answer']
    except Exception as e:
        return f"Error: {str(e)}"

# ✅ Create Gradio Interface
interface = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.Textbox(lines=3, placeholder="Enter context (passage) here...", label="Context"),
        gr.Textbox(lines=1, placeholder="Enter your question...", label="Question")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="📖 AI Question Answering System",
    description="Provide a passage and ask a question about it. The model will extract the best possible answer!",
    theme="compact",
    live=True,  # Real-time updates
)

# ✅ Launch Gradio
interface.launch(share=True)


Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/gradio/blocks.py:1098: UserWarning: Cannot load compact. Caught Exception: 404 Client Error: Not Found for url: https://huggingface.co/api/spaces/compact (Request ID: Root=1-67ad1da2-0d627ca8682b57184aa29826;959b287a-6260-405a-a34f-aefc183ea95d)

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8b6628dcc2c4379d9.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
